In [ ]:
!pip -q install langchain
!pip -q install openai
!pip -q install python-dotenv
!pip -q install langchain_openai
!pip -q install tiktoken
!pip -q install langchain_experimental
!pip -q install langchain[all]


In [ ]:
CSV_PATH = r"Projects\ADHD2\analysis_results\MBERT_covercheck_uncover.csv"   # 질문/표현 데이터
COL_NAME = "question_full"       

EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"  
TARGET_TOPICS = 5         
MIN_TOPIC_SIZE = 30         

import sys, subprocess
for pk in ["bertopic", "sentence-transformers", "umap-learn", "hdbscan", "pandas", "numpy", "scikit-learn", "openai", "python-dotenv"]:
    try:
        __import__(pk.split("[")[0].replace("-", "_"))
    except Exception:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pk])

import re, os, json, numpy as np, pandas as pd
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True) 

def split_sents_ko(text):
    if not text: return []
    text = re.sub(r"https?://\S+|www\.\S+", " ", str(text))
    text = re.sub(r"\s+", " ", text).strip()
    sents = re.split(r'(?<=[\.\?\!])\s+|(?<=요)\s+|(?<=다)\s+|\n+', text)
    return [s.strip() for s in sents if len(s.strip()) >= 8]

try:
    df = pd.read_csv(CSV_PATH)
except UnicodeDecodeError:
    df = pd.read_csv(CSV_PATH, encoding="cp949")
assert COL_NAME in df.columns, f"CSV에 '{COL_NAME}' 컬럼이 없습니다."

raw_texts = df[COL_NAME].fillna("").astype(str).tolist()
sentences = []
for t in raw_texts:
    sentences.extend(split_sents_ko(t))

sentences = list(dict.fromkeys([s for s in sentences if len(s) >= 8]))
print("문장 수:", len(sentences))
if len(sentences) < 10:
    raise ValueError("문장이 너무 적습니다(>=10 권장).")

topic_info = topic_model.get_topic_info()

In [ ]:
# =========================================================
# LimTopic Pipeline
# 1) BERTopic first
# 2) GPT-4o mini topic labeling/summary second
# =========================================================

import os
import time
import json
import ast
import pandas as pd

from pathlib import Path
from dotenv import load_dotenv

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from openai import OpenAI

PROJECT_DIR = Path(r"Projects\ADHD2")
load_dotenv(PROJECT_DIR / ".env", override=True)
OUT_DIR = PROJECT_DIR / "Data Availability/ valid"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_BERTOPIC = OUT_DIR / "MBERT_uncover_BERTopic.csv"
OUT_ASSIGN = OUT_DIR / "MBERT_uncover_sentences_LimTopic.csv"
OUT_FINAL = OUT_DIR / "MBERT_uncover_LimTopic.csv"

OPENAI_API_KEY = (os.getenv("OPENAI_API_KEY") or "").strip()

if not OPENAI_API_KEY:
    raise RuntimeError("OPENAI_API_KEY가 없습니다. .env 파일을 확인하세요.")

client = OpenAI(api_key=OPENAI_API_KEY)

# =========================================================
# 1. Parameters
# =========================================================

EMBED_MODEL = os.getenv(
    "EMBED_MODEL",
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

LLM_MODEL = os.getenv(
    "LLM_MODEL",
    "gpt-4o-mini"
)

MIN_TOPIC_SIZE = int(os.getenv("MIN_TOPIC_SIZE", "4"))
TARGET_TOPICS = int(os.getenv("TARGET_TOPICS", "30"))

print("=" * 70)
print("[Pipeline] BERTopic first, GPT labeling second")
print(f"[INPUT]        {INPUT_PATH}")
print(f"[TEXT_COL]     {TEXT_COL}")
print(f"[EMBED_MODEL]  {EMBED_MODEL}")
print(f"[LLM_MODEL]    {LLM_MODEL}")
print(f"[MIN_TOPIC]    {MIN_TOPIC_SIZE}")
print(f"[TARGET_TOPIC] {TARGET_TOPICS}")
print("=" * 70)

# =========================================================
# 2. Load Sentences
# =========================================================

df_input = pd.read_csv(INPUT_PATH, encoding="utf-8-sig")

if TEXT_COL not in df_input.columns:
    raise ValueError(f"'{TEXT_COL}' 컬럼이 없습니다. 현재 컬럼: {list(df_input.columns)}")

sentences = (
    df_input[TEXT_COL]
    .dropna()
    .astype(str)
    .map(lambda x: x.strip())
)

sentences = sentences[sentences.str.len() > 0].tolist()

if len(sentences) == 0:
    raise ValueError("분석할 문장이 없습니다.")

print(f"[N_SENTENCES] {len(sentences)}")

# =========================================================
# 3. BERTopic
# =========================================================

embedder = SentenceTransformer(EMBED_MODEL)

umap_model = UMAP(
    n_neighbors=10,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42,
)

topic_model = BERTopic(
    embedding_model=embedder,
    umap_model=umap_model,
    language="korean",
    calculate_probabilities=True,
    min_topic_size=MIN_TOPIC_SIZE,
    n_gram_range=(1, 2),
    top_n_words=15,
    verbose=True,
)

topics, probs = topic_model.fit_transform(sentences)

# =========================================================
# 4 Save BERTopic Outputs
# =========================================================

topic_info = topic_model.get_topic_info()
topic_info.to_csv(OUT_BERTOPIC, index=False, encoding="utf-8-sig")
print(f"[SAVED] {OUT_BERTOPIC}")

assign_df = pd.DataFrame({
    "sentence": sentences,
    "topic": topics,
})

if probs is not None:
    try:
        assign_df["probability"] = probs.max(axis=1)
    except Exception as e:
        print(f"[WARN] probability extraction failed: {e}")

assign_df.to_csv(OUT_ASSIGN, index=False, encoding="utf-8-sig")
print(f"[SAVED] {OUT_ASSIGN}")

# =========================================================
# 5 Build Topic Documents / Keywords for LLM
# =========================================================

docs_by_topic = (
    assign_df
    .groupby("topic")["sentence"]
    .apply(lambda x: list(x)[:10])
    .to_dict()
)

keywords_by_topic = {}

for topic_id in topic_info["Topic"]:
    if topic_id == -1:
        continue

    topic_words = topic_model.get_topic(topic_id)

    if topic_words:
        keywords_by_topic[topic_id] = [word for word, score in topic_words[:15]]
    else:
        keywords_by_topic[topic_id] = []

topic_rows = []

for _, row in topic_info.iterrows():
    topic_id = row["Topic"]

    if topic_id == -1:
        continue

    docs = docs_by_topic.get(topic_id, [])
    keywords = keywords_by_topic.get(topic_id, [])

    topic_rows.append({
        "Topic": topic_id,
        "Count": row.get("Count", None),
        "Name": row.get("Name", ""),
        "Keywords": keywords,
        "Representative_Docs": docs,
    })

limtopic_df = pd.DataFrame(topic_rows)

# =========================================================
#6 GPT-4o mini Topic Title / Description
# =========================================================

def normalize_text_bundle(text):
    if isinstance(text, str):
        s = text.strip()

        if s.startswith("[") and s.endswith("]"):
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, list):
                    return "\n".join([str(x) for x in parsed])
            except Exception:
                pass

        return s

    if isinstance(text, list):
        return "\n".join([str(x) for x in text])

    return ""


def gen_topic_title_description(docs, keywords):
    docs_text = normalize_text_bundle(docs)
    keywords_text = ", ".join(keywords) if isinstance(keywords, list) else str(keywords)

    if not docs_text.strip():
        return {
            "llm_title": "",
            "llm_description": "",
        }

    system = (
        "당신은 소비자 건강정보를 분석하는 토픽 모델링 전문가입니다. "
        "문서들을 요약하는 것이 아니라, 문서들에 공통적으로 내재된 잠재 주제를 도출하세요. "
        "반드시 JSON 한 줄로만 답하세요."
    )

    user = f"""
아래에는 하나의 토픽을 대표하는 여러 개의 문서와 핵심 키워드가 제시됩니다.

당신의 목표는 문서를 요약하는 것이 아니라,
문서들에 공통적으로 내재되어 있는 '잠재 주제(latent topic)'를 발견하는 것입니다.

다음 원칙을 반드시 따르세요.

[분석 원칙]

1. 대표 문서를 가장 우선적으로 분석하세요.
   키워드는 문서를 이해하기 위한 참고자료로만 활용하세요.

2. 문서에서 반복되는 단어를 그대로 사용하는 것이 아니라,
   문서들이 공통적으로 의미하는 핵심 개념을 추론하세요.

3. 서로 다른 표현이라도 동일한 의미를 나타낸다면
   하나의 의미로 통합하여 해석하세요.

4. 너무 일반적인 표현(예: 건강관리, 치료, 질병, 정보 등)은
   토픽명으로 사용하지 마세요.

5. 토픽명은 소비자의 unmet needs 또는 관심사를 가장 잘 설명할 수 있도록
   구체적이고 간결하게 작성하세요.
   토픽명을 작성하기 전에 먼저 문서들의 공통된 의미를 충분히 추론한 후
   가장 적절한 하나의 주제를 선택하세요.

6. 설명은 이 토픽이 의미하는 핵심 내용을 한 문장으로 작성하세요.

7. 문서에 없는 내용을 추론하거나 추가하지 마세요.

--------------------------------------------------

대표 문서:
{docs_text}

--------------------------------------------------

핵심 키워드:
{keywords_text}

--------------------------------------------------

다음 JSON 형식으로만 출력하세요.

{{"title": "한국어 토픽명 3~7단어", "description": "토픽을 대표하는 한 문장 설명"}}
"""

    try:
        resp = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
            max_tokens=500,
            temperature=0.2,
        )

        content = resp.choices[0].message.content

        if not content:
            return {
                "llm_title": "",
                "llm_description": "ERROR: empty LLM response",
            }

        content = content.strip()

        if content.startswith("```"):
            content = content.replace("```json", "").replace("```", "").strip()

        parsed = json.loads(content)

        return {
            "llm_title": parsed.get("title", ""),
            "llm_description": parsed.get("description", ""),
        }

    except Exception as e:
        return {
            "llm_title": "",
            "llm_description": f"ERROR: {e}",
        }


titles = []
descriptions = []

for idx, row in limtopic_df.iterrows():
    out = gen_topic_title_description(
        docs=row["Representative_Docs"],
        keywords=row["Keywords"],
    )

    titles.append(out["llm_title"])
    descriptions.append(out["llm_description"])

    print(f"[LLM] topic={row['Topic']} title={out['llm_title']}")
    time.sleep(1.5)

limtopic_df["llm_title"] = titles
limtopic_df["llm_description"] = descriptions

# =========================================================
# 7 Save Final LimTopic Output
# =========================================================

limtopic_df.to_csv(
    OUT_FINAL,
    index=False,
    encoding="utf-8-sig",
)

print(f"[SAVED] {OUT_FINAL}")